# 02 · Text OCR

- load detected bubble crops from notebook 01
- extract text.

## 1. Load artifacts from bubble detector
Pull the annotated image and bounding boxes generated in notebook 01.

In [1]:
from pathlib import Path
import json
from PIL import Image

ARTIFACT_DIR = Path('artifacts')
IMAGE_PATH = ARTIFACT_DIR / 'latest_text_bubble_detection.png'
BOXES_PATH = ARTIFACT_DIR / 'latest_text_bubble_boxes.json'

if not IMAGE_PATH.exists() or not BOXES_PATH.exists():
    print('Artifacts not found. Run `01-bubble-detection.ipynb` to generate detections first.')
else:
    annotated_image = Image.open(IMAGE_PATH).convert('RGB')
    boxes_payload = json.loads(BOXES_PATH.read_text())
    print(f'Loaded {len(boxes_payload)} candidate regions from {BOXES_PATH}')


Loaded 2 candidate regions from artifacts/latest_text_bubble_boxes.json


## 2. Slice image into candidate regions
Crop each bounding box from the annotated image so we can feed them to the OCR engine.

In [2]:
bubble_crops = []
if 'annotated_image' in globals():
    width, height = annotated_image.size
    for idx, entry in enumerate(boxes_payload, start=1):
        box = entry['box']
        left, top, right, bottom = [int(round(v)) for v in box]
        left = max(0, left)
        top = max(0, top)
        right = min(width, right)
        bottom = min(height, bottom)
        crop = annotated_image.crop((left, top, right, bottom))
        bubble_crops.append({'image': crop, 'score': entry.get('score', 0.0), 'box': [left, top, right, bottom]})
    print(f'Prepared {len(bubble_crops)} crops for OCR.')
else:
    print('Skipping crop step: artifacts missing.')


Prepared 2 crops for OCR.


## 3. OCR each region
Use `ocrmac` to pull text from each detected bubble.

In [6]:
from ocrmac import ocrmac

FINAL_RESULT = []

if 'bubble_crops' in globals() and bubble_crops:
  for idx, entry in enumerate(bubble_crops, start=1):
    result = annotations = ocrmac.livetext_from_image(entry['image'])
    text = result.strip() if isinstance(result, str) else str(result)
    line = (" ".join(el[0] for el in annotations))
    print(f"Bubble #{idx}:\n\n{line} \n")
    FINAL_RESULT.append(line)
else:
    print('No crops available for OCR. Ensure previous steps ran successfully.')

Bubble #1:

Je suis retournée seule dans ma chambre de verre. Je vous ai tout donné, mon corps, mor intimité, le sexe, ma solitude parfois. La routine aussi. Mais cette tristesse, je crois qu'elle n'a rien d'excitant ni d'intéressant. Et je ne veux pas la partager, avec qui que ce soit. Je ferme le site. 

Bubble #2:

Je l'ai perdu. Je le craignais dès le départ. Qu'un jour ou l'autre il me trouve trop vieille, qu'il rencontre une autre femme, plus jeune e séduisante que moi. Mais non, ça ne s'est pas passé comme ça. Et ça fait encore plus mal car tout est de ma faute. 



## 5. Save OCR output
Persist recognised bubble text for downstream notebooks.

In [4]:
ARTIFACT_DIR = Path('artifacts')
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
payload = {'bubbles': FINAL_RESULT}
target_path = ARTIFACT_DIR / 'latest_text_bubble_text.json'
target_path.write_text(json.dumps(payload, indent=2))
print(f'Saved bubble text to {target_path}')

Install ocrmac (see step 1) before saving OCR output.
No OCR text to save.
